In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# 1. Load Silver Products
# ============================================================

silver_products_df = spark.table(
    "customer360_dev.silver.products"
)

target_table = "customer360_dev.gold.dim_product"

In [0]:
product_window = (
    Window
    .partitionBy("product_id")
    .orderBy(
        F.col("updated_at").desc_nulls_last(),
        F.col("_ingestion_timestamp").desc_nulls_last()
    )
)

latest_product_df = (
    silver_products_df
    .withColumn(
        "_row_number",
        F.row_number().over(product_window)
    )
    .filter(
        F.col("_row_number") == 1
    )
    .drop("_row_number")
)

In [0]:
dim_product_df = (
    latest_product_df

    .select(
        "product_id",
        "product_name",
        "category_code",
        "subcategory_code",
        "brand_name",
        "unit_price",
        "cost_price",
        "product_status",
        "launch_date",
        "discontinued_date",
        "supplier_id",
        "currency_code",
        "created_at",
        "updated_at"
    )

    .withColumn(
        "_gold_created_at",
        F.current_timestamp()
    )

    .withColumn(
        "_gold_updated_at",
        F.current_timestamp()
    )
)

In [0]:
(
    dim_product_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        target_table
    )
)

print(
    "dim_product created successfully."
)

dim_product created successfully.


In [0]:
silver_product_count = (
    silver_products_df
    .select("product_id")
    .distinct()
    .count()
)

gold_product_count = (
    spark.table(target_table)
    .count()
)

print(
    f"Silver distinct products : {silver_product_count:,}"
)

print(
    f"Gold dim_product rows    : {gold_product_count:,}"
)

Silver distinct products : 149,400
Gold dim_product rows    : 149,400


In [0]:
duplicate_product_ids = (
    spark.table(target_table)
    .groupBy("product_id")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

print(
    "Duplicate product IDs:",
    duplicate_product_ids
)

Duplicate product IDs: 0


In [0]:
display(
    spark.table(target_table)
    .select(
        "product_id",
        "product_name",
        "category_code",
        "subcategory_code",
        "brand_name",
        "unit_price",
        "cost_price",
        "product_status",
        "currency_code"
    )
    .limit(50)
)

product_id,product_name,category_code,subcategory_code,brand_name,unit_price,cost_price,product_status,currency_code
PROD000000001,Product 1,SPORTS,SUBCAT02,URBAN,101.37,68.93,ACTIVE,INR
PROD000000002,Product 2,GROCERY,SUBCAT03,ZEN,102.74,69.86,ACTIVE,INR
PROD000000003,Product 3,BEAUTY,SUBCAT04,PRIME,104.11,70.79,ACTIVE,INR
PROD000000004,Product 4,HOME,SUBCAT05,AURA,105.48,71.73,ACTIVE,INR
PROD000000005,Product 5,FASHION,SUBCAT06,NEXUS,106.85,72.66,ACTIVE,INR
PROD000000006,Product 6,ELECTRONICS,SUBCAT07,NOVA,108.22,73.59,ACTIVE,INR
PROD000000007,Product 7,SPORTS,SUBCAT08,URBAN,109.59,74.52,ACTIVE,INR
PROD000000008,Product 8,GROCERY,SUBCAT09,ZEN,110.96,75.45,ACTIVE,INR
PROD000000009,Product 9,BEAUTY,SUBCAT10,PRIME,112.33,76.38,ACTIVE,INR
PROD000000010,Product 10,HOME,SUBCAT11,AURA,113.70,77.32,ACTIVE,INR
